# Beginner tour 2: Your first experiment

In notebook 1 you met the data: 21,044 real molecules, their docking scores, and their
embeddings. Now you'll run **the project's real experiment** on them, using exactly the code
the researchers use on 2 million molecules.

**What you'll do:**
1. Understand the idea, *active learning*, in plain words
2. See how a researcher starts an experiment from the command line
3. Run it, then look at every file it saves
4. Read the heart of the code, about 30 lines, step by step
5. Change things and see what happens

Run the cells from top to bottom with **Shift + Enter**.

---
## 1. The idea: active learning

Remember the problem: we want the best 100 of 21,044 molecules, but pretend docking is so
expensive that we can only afford to dock a few. **Active learning** is a loop:

```
   ┌──────────────────────────────────────────────────────────────────────┐
   │                                                                      │
 dock a small        teach a quick         let it guess         dock the        add them to
 random handful ──►  model with what ───►  every undocked  ───► best-looking ──► what we know
 (round 0)           we know so far        molecule             guesses
                     ("the surrogate")                          (one "batch")        │
                          ▲                                                          │
                          └─────────────────── repeat for 5 rounds ◄─────────────────┘
```

At the end we count how many of the true best 100 we docked along the way. That's the
**recall**: 50% recall means we found half of them.

The settings for this notebook (the same *proportions* as the real experiments, scaled down):

| Setting | Value | Meaning |
|---|---|---|
| start | 210 molecules | 1% of the pool, chosen at random |
| batch | 210 molecules per round | another 1% each round |
| rounds | 5 | so 210 + 5 × 210 = **1,260 docked in total**, 6% of the pool |
| goal | the best 100 | top 0.5% |

**Question to think about first:** if we docked 1,260 molecules *at random*, what recall
would we expect? (Answer in section 5.)

In [ ]:
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tour import *

import run_experiment as rx
register_mini(rx)            # makes our mini dataset available as --dataset MINI
print("ready")

---
## 2. How a researcher starts an experiment

Researchers don't click buttons. They type a **command** in a terminal. Here's one copied
from the project's cluster scripts (`slurm/al_runs/enamine/submit_al_runs.sh`):

```bash
python run_experiment.py --mode mve --surrogate ensemble --backbones grover molformer unimol \
    --acq greedy --init-size 2104 --batch-size 2104 --n-rounds 5 --topk 1000 \
    --run-dir runs/fusion_greedy_frac0.001
```

* `python run_experiment.py` means "run this Python file".
* Everything after it are **options** (also called *flags* or *arguments*). `--n-rounds 5`
  means "the option `n-rounds` is 5".

🐍 **Python corner: running terminal commands from a notebook.** A line starting with `!` is
sent to the terminal instead of Python. `--help` asks a program to list its options. Here are
the first ones:

In [ ]:
!cd {AL_REPO} && python run_experiment.py --help 2>/dev/null | head -32

Inside the program, a function called `parse_args()` reads those words and turns them into
an object called `args`, with one **attribute** per option. (An attribute is a named value
stored on an object, read with a dot: `args.n_rounds`.) Note that the dashes become
underscores: `--n-rounds` turns into `args.n_rounds`.

🐍 **Python corner: `sys.argv`.** The words of the command line are stored in a list called
`sys.argv`. In a notebook there's no command line, so we **fill that list in ourselves**, and
then the program can't tell the difference.

In [ ]:
run_dir = new_run_dir()       # an empty folder for this experiment's output

command = [
    "run_experiment.py",
    "--dataset", "MINI",                 # our 21,044 real molecules
    "--mode", "mve",                     # use pre-computed embeddings
    "--surrogate", "single",             # the simplest model: one backbone
    "--backbone", "unimol2",             # ...the Uni-Mol2 one (reads 3D shape)
    "--acq", "greedy",                   # pick the molecules with the best guesses
    "--init-size", "210", "--batch-size", "210", "--n-rounds", "5",
    "--topk", "100",                     # success = finding the best 100
    "--run-dir", str(run_dir),           # where to save results
]
sys.argv = command
args = rx.parse_args()
print("args.surrogate =", args.surrogate)
print("args.n_rounds  =", args.n_rounds)
print("args.seed      =", args.seed, "  <- we didn't give one, so it's the default")

---
## 3. Run it!

`rx.main()` is the function the command line runs. It reads `sys.argv`, loads the data,
builds the model, and runs the loop. Watch what it prints: one line per round.

In [ ]:
rx.main()

You may also see some red text, warnings from libraries the project imports (about `tqdm`,
`ipywidgets`, and so on). They're harmless. `device=cuda` means it's using a GPU; on a
computer without one it says `device=cpu` and runs a little slower.

How to read that:
* `[oracle]`, `[EmbeddingFeaturizer]`, `[pool]`: loading the scores and the embeddings.
* `[init] 210 random molecules best=...`: round 0, the random start.
* `Round 01/5 labeled=420 ... top-100 recall=...`: after each round, how many are docked and
  what fraction of the best 100 we've found so far.

**You just ran a real active-learning campaign.** On the full 2.1-million-molecule pool,
the same code runs on a GPU cluster.

---
## 4. What did it save?

🐍 **Python corner: listing files.** `run_dir.rglob("*")` finds everything inside a folder
(the `r` is for *recursive*: sub-folders too).

In [ ]:
for p in sorted(run_dir.rglob("*")):
    print(p.relative_to(run_dir))

| File | What's in it |
|---|---|
| `history.json` | one summary per round: the learning curve |
| `all_explored_final.csv` | every molecule we docked, best first |
| `iter_N/state.json`, `iter_N/scores.pkl` | a **checkpoint** after round N, so a crashed run can continue (`--resume`) |

🐍 **Python corner: JSON.** JSON is a common text format for lists and dictionaries.
`json.loads(text)` turns it into Python lists and dicts.

In [ ]:
history = json.loads((run_dir / "history.json").read_text())
pd.DataFrame(history)          # a list of dicts makes a neat table

The best molecules this campaign found:

In [ ]:
found = pd.read_csv(run_dir / "all_explored_final.csv")
print(len(found), "molecules docked in total")
draw(found["smiles"][:8], labels=found["score"][:8])

---
## 5. Is that good? Compare with random

To judge active learning we need a **baseline**: the same budget, spent at random.

🐍 **Python corner: writing your own function.** `def name(inputs):` defines a function and
`return` gives back the answer. Ours docks random molecules and counts how many of the best
100 it hits.

In [ ]:
scores, embeddings = load_mini()
best100 = set(scores.nsmallest(100, "score")["smiles"])     # a set: fast "is it in here?" checks

def random_recall(n_docked, seed):
    rng = np.random.default_rng(seed)
    picked = rng.choice(len(scores), n_docked, replace=False)
    return sum(s in best100 for s in scores["smiles"][picked]) / 100

print("random, 1,260 molecules, 5 tries:", [random_recall(1260, s) for s in range(5)])

Random docking finds about **6%** of the best 100, because it docks 6% of everything. Active
learning found roughly **15× more**, with the same budget. That gap is the whole point of
the project.

Let's draw both as curves:

In [ ]:
n_labeled = [210] + [h["n_labeled"] for h in history]
al_recall = [0] + [h["topk_recall"] for h in history]
plt.plot(n_labeled, al_recall, marker="o", label="active learning (Uni-Mol2)")
plt.plot(n_labeled, [n / len(scores) for n in n_labeled], marker="o", label="random (expected)")
plt.xlabel("molecules docked"); plt.ylabel("fraction of the best 100 found"); plt.ylim(0, 1)
plt.legend(); plt.show()

(The starting point is drawn at 0 for simplicity. The random 210 may already contain one or
two of the best 100.)

---
## 6. Reading the heart of the code

Everything you saw happens in one function, `MVEExplorer.run` in `run_experiment.py`. Let's
print it. (`inspect.getsource` shows the code of any function.) Don't worry about
understanding every line. The table below explains it in plain words.

In [ ]:
import inspect
source = inspect.getsource(rx.MVEExplorer.run)
lines = source.splitlines()
print("\n".join(lines[:38]))

| Code | Plain words |
|---|---|
| `for rnd in range(self._start_round, self.n_rounds):` | repeat once per round |
| `idx = list(self.labeled_idx)` | the molecules docked so far |
| `ys = self._sign * np.array([...])` | their scores, **times −1**, so that *bigger = better* (makes "pick the best" mean "pick the biggest") |
| `self.model.train(xs, ys, ...)` | **teach the model** what we know |
| `mask = np.ones(n, bool)` … `mask[i] = False` | mark which molecules are *not* docked yet |
| `mu, var = _chunked_get_means_and_vars(...)` | **the model guesses** every undocked molecule: `mu` = guess, `var` = how unsure it is |
| `scores = self.acq_fn(mu)` | turn guesses into a ranking (`greedy`: just use the guess) |
| `top_local = np.argsort(scores)[::-1][: self.batch_size]` | sort, biggest first, take the top 210 |
| `self.labeled_scores[smi] = self.oracle[smi]` | **"dock" them**: look up the true score |
| `recall = self._recall()` | how many of the best 100 we've found |
| `self._checkpoint(rnd + 1, record)` | save `iter_N/` |

🐍 **Python corner: `self`.** Inside a class (a bundle of data plus functions),
`self.something` means "the `something` that belongs to this object": its model, its
labeled set, its batch size. Notebook 5 explains classes properly.

That's the whole algorithm. The rest of the project is about doing these steps **better**
(notebooks 3–4, 6) and **bigger** (notebook 7).

---
## 7. Experiments you can run

To make experimenting easy, wrap "build the command, run, read history" into one function.

🐍 **Python corner: default values and `*`.** `def run(backbone="unimol2", seed=42, ...)`
means that if you don't pass `seed`, it's 42. `*extra` collects any additional words into a
list. `contextlib.redirect_stdout` hides the per-round printing so the output stays tidy.

In [ ]:
import contextlib, io

def run(backbone="unimol2", seed=42, init=210, batch=210, rounds=5, acq="greedy"):
    out = new_run_dir()
    sys.argv = ["run_experiment.py", "--dataset", "MINI", "--mode", "mve",
                "--surrogate", "single", "--backbone", backbone, "--acq", acq,
                "--init-size", str(init), "--batch-size", str(batch), "--n-rounds", str(rounds),
                "--topk", "100", "--seed", str(seed), "--run-dir", str(out)]
    with contextlib.redirect_stdout(io.StringIO()):
        rx.main()
    return [h["topk_recall"] for h in json.loads((out / "history.json").read_text())]

print("seed 42:", run(seed=42))

**Experiment A: luck.** The random start differs with the **seed** (the starting point of
the random-number generator). How much does the final recall move?

In [ ]:
finals = [run(seed=s)[-1] for s in [42, 43, 44, 45, 46]]
print("final recall, 5 seeds:", finals)
print(f"mean {np.mean(finals):.2f}, spread (std) {np.std(finals):.2f}")

The researchers always run **5 seeds** and report mean ± spread. A difference smaller than
the spread is probably luck, not a real improvement.

**Experiment B: the shape of the budget.** Same total, split differently: one big batch,
or many small ones. First with our usual budget of 1,260 molecules:

In [ ]:
for init, batch, rounds in [(210, 525, 2), (210, 210, 5), (210, 105, 10)]:
    finals = [run(init=init, batch=batch, rounds=rounds, seed=s)[-1] for s in [42, 43, 44]]
    print(f"{rounds:2d} rounds x {batch:3d} (total {init + batch * rounds}):  mean final recall {np.mean(finals):.2f}")

Hardly any difference. With 1,260 molecules the model finds almost all of the best 100
however we split the budget: we've hit the **ceiling**, so there's nothing left to improve.

To see an effect, make the budget **tight**: only 420 molecules, 2% of the pool.

In [ ]:
for init, batch, rounds in [(105, 315, 1), (105, 105, 3), (105, 35, 9)]:
    finals = [run(init=init, batch=batch, rounds=rounds, seed=s)[-1] for s in [42, 43, 44]]
    print(f"{rounds:2d} rounds x {batch:3d} (total {init + batch * rounds}):  mean final recall {np.mean(finals):.2f}")

Now it matters a lot. **More, smaller rounds** give the model more chances to learn from
what it just docked before choosing again. Two lessons in one:

1. An experiment can only show a difference if it isn't already at the ceiling. Choose
   settings where there's room to improve.
2. The project found the same thing on the full 2.1M pool: 20 rounds beat 5 rounds at the
   same budget (`al-molecular/docs/adaptive_acq_results.md`). In real life more rounds also
   cost more time, because each round waits for its docking to finish.

---
## Summary

* **Active learning** = dock a few → teach a model → let it pick the next batch → repeat.
* An experiment is started with a **command**. `parse_args()` turns the words into `args`,
  and `main()` builds everything and runs the loop.
* Results land in a run folder: `history.json` (the curve), `all_explored_final.csv` (what
  was found), `iter_N/` (checkpoints).
* With 6% of the pool docked, the model found far more of the best 100 than random's 6%.
* Always compare **several seeds** against a **random baseline**.

🐍 **Python you met:** `!` terminal commands, `sys.argv`, objects with attributes (`args.x`),
JSON, `pd.DataFrame`, `set`, writing functions with defaults, `inspect.getsource`.

## Try it yourself
1. Run `run(backbone="molformer")` and `run(backbone="mhgged")`. Which backbone finds the most? (Notebook 4 explains why.)
2. Run with `acq="ucb"` instead of `"greedy"`. Any difference? (Notebook 4 again.)
3. Change `--topk` to 1000 in `run`. What happens to recall, and why?

**Next:** `3_how_the_model_learns.ipynb`, opening up the surrogate model.